# Práctica integradora de la Unidad 3

## Comparación de bordes, segmentación y morfología

**Nombre:** ____________________  
**Fecha:** ____________________

Este notebook permite comparar, con el mismo flujo, las siguientes técnicas:

- detección de bordes con Canny;
- umbralización global con Otsu;
- umbralización adaptativa;
- agrupamiento k-means sobre intensidades;
- apertura y cierre morfológico;
- segmentación por color para el caso de frutas.

La comparación incluye imágenes de objetos, una radiografía, muestras clásicas de OpenCV, imágenes de scikit-image y `carta.jpg`. La meta es decidir qué método resulta más adecuado según el objetivo, no buscar un método universalmente mejor.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import data

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10
carpeta = Path('imagenes')
print('OpenCV:', cv2.__version__)
print('Carpeta de imágenes:', carpeta.resolve())

## 1. Banco de imágenes

Las imágenes locales se buscan en `Unidad3/imagenes`. `astronaut` y `coins` se cargan desde `skimage.data`.

In [ ]:
def cargar_banco():
    banco = {}
    archivos = ['frutas.jpg', 'radiografia.jpg', 'messi5.jpg', 'lena.jpg', 'carta.jpg']
    for nombre in archivos:
        ruta = carpeta / nombre
        imagen_bgr = cv2.imread(str(ruta), cv2.IMREAD_COLOR)
        if imagen_bgr is None:
            raise FileNotFoundError(f'No se encontró la imagen: {ruta}')
        banco[nombre] = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)
    banco['astronaut'] = data.astronaut()
    banco['coins'] = data.coins()
    return banco

banco = cargar_banco()
fig, axs = plt.subplots(2, 4, figsize=(15, 8))
for ax, (nombre, imagen) in zip(axs.ravel(), banco.items()):
    ax.imshow(imagen, cmap='gray' if imagen.ndim == 2 else None)
    ax.set_title(nombre, fontweight='bold')
    ax.axis('off')
axs.ravel()[-1].axis('off')
plt.tight_layout()
plt.show()

## 2. Flujo común de comparación

Para cada imagen se calcula una representación en gris, Canny, Otsu, umbral adaptativo, k-means con `K=3` y una máscara refinada con apertura y cierre. La máscara del grupo más oscuro es solo una hipótesis de trabajo y debe revisarse visualmente.

In [ ]:
def comparar_metodos(imagen_rgb):
    if imagen_rgb.ndim == 2:
        gris = imagen_rgb.copy()
        original = cv2.cvtColor(imagen_rgb, cv2.COLOR_GRAY2RGB)
    else:
        original = imagen_rgb.copy()
        gris = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2GRAY)

    suavizada = cv2.GaussianBlur(gris, (5, 5), 1.0)
    bordes = cv2.Canny(suavizada, 60, 140)
    otsu_valor, otsu = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adaptativa = cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 5)

    datos = gris.reshape((-1, 1)).astype(np.float32)
    criterio = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 0.1)
    compactness, etiquetas, centros = cv2.kmeans(datos, 3, None, criterio, 10, cv2.KMEANS_PP_CENTERS)
    centros = centros.ravel()
    etiquetas_2d = etiquetas.reshape(gris.shape)
    cuantizada = centros[etiquetas.ravel()].reshape(gris.shape).astype(np.uint8)
    grupo_oscuro = int(np.argmin(centros))
    mascara_kmeans = (etiquetas_2d == grupo_oscuro).astype(np.uint8) * 255

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    refinada = cv2.morphologyEx(mascara_kmeans, cv2.MORPH_OPEN, kernel)
    refinada = cv2.morphologyEx(refinada, cv2.MORPH_CLOSE, kernel)
    return {'original': original, 'gris': gris, 'bordes': bordes, 'otsu': otsu,
            'adaptativa': adaptativa, 'kmeans': cuantizada, 'refinada': refinada,
            'otsu_valor': float(otsu_valor), 'centros': np.sort(centros),
            'compactness': float(compactness)}

def mostrar_comparacion(nombre, r):
    vistas = [(r['original'], 'Original', None), (r['gris'], 'Gris', 'gray'),
             (r['bordes'], 'Canny', 'gray'), (r['otsu'], f"Otsu (T={r['otsu_valor']:.0f})", 'gray'),
             (r['adaptativa'], 'Adaptativa', 'gray'), (r['kmeans'], 'K-means K=3', 'gray'),
             (r['refinada'], 'K-means + morfología', 'gray')]
    fig, axs = plt.subplots(2, 4, figsize=(14, 8))
    for ax, (imagen, titulo, mapa) in zip(axs.ravel(), vistas):
        ax.imshow(imagen, cmap=mapa)
        ax.set_title(titulo, fontweight='bold')
        ax.axis('off')
    axs.ravel()[-1].axis('off')
    fig.suptitle(nombre, fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

resultados = {}
for nombre, imagen in banco.items():
    resultados[nombre] = comparar_metodos(imagen)
    mostrar_comparacion(nombre, resultados[nombre])

## 3. Medidas simples

Estas medidas no sustituyen una máscara de referencia, pero ayudan a detectar resultados extremos: demasiados píxeles blancos, muchas componentes pequeñas o una región dominante muy grande.

In [ ]:
def medidas_mascara(mascara):
    binaria = (mascara > 0).astype(np.uint8)
    componentes, _, estadisticas, _ = cv2.connectedComponentsWithStats(binaria, connectivity=8)
    area_mayor = int(np.max(estadisticas[1:, cv2.CC_STAT_AREA])) if componentes > 1 else 0
    return 100 * float(np.mean(binaria)), componentes - 1, area_mayor

filas = []
for nombre, r in resultados.items():
    for metodo in ['otsu', 'adaptativa', 'refinada']:
        porcentaje, componentes, area_mayor = medidas_mascara(r[metodo])
        filas.append({'imagen': nombre, 'metodo': metodo, 'porcentaje_blanco': porcentaje,
                      'componentes': componentes, 'area_mayor': area_mayor})

tabla = pd.DataFrame(filas)
tabla.round(2)

## 4. Segmentación por color: frutas

Para `frutas.jpg`, la intensidad en gris puede perder diferencias importantes entre colores. HSV separa tono, saturación y brillo; por eso permite probar una regla basada en saturación y también k-means sobre color. La segmentación no identifica automáticamente una fruta: primero produce regiones y después se deben calcular descriptores o entrenar un clasificador.

In [ ]:
frutas_rgb = banco['frutas.jpg']
frutas_hsv = cv2.cvtColor(frutas_rgb, cv2.COLOR_RGB2HSV)
pixeles_hsv = frutas_hsv.reshape((-1, 3)).astype(np.float32)
criterio_color = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 0.2)
_, etiquetas, centros = cv2.kmeans(pixeles_hsv, 4, None, criterio_color, 12, cv2.KMEANS_PP_CENTERS)
etiquetas = etiquetas.reshape(frutas_hsv.shape[:2])
centros = centros.astype(np.float32)
grupos_saturados = np.flatnonzero(centros[:, 1] > np.median(centros[:, 1]))
mascara_color = np.isin(etiquetas, grupos_saturados).astype(np.uint8) * 255
kernel_color = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
mascara_color = cv2.morphologyEx(mascara_color, cv2.MORPH_OPEN, kernel_color)
mascara_color = cv2.morphologyEx(mascara_color, cv2.MORPH_CLOSE, kernel_color, iterations=2)

fig, axs = plt.subplots(1, 4, figsize=(15, 4))
axs[0].imshow(frutas_rgb); axs[0].set_title('RGB')
axs[1].imshow(frutas_hsv[:, :, 0], cmap='hsv', vmin=0, vmax=179); axs[1].set_title('Tono HSV')
axs[2].imshow(mascara_color, cmap='gray'); axs[2].set_title('Segmentación por color')
axs[3].imshow(frutas_rgb); axs[3].imshow(mascara_color, cmap='Reds', alpha=0.35); axs[3].set_title('Superposición')
for ax in axs: ax.axis('off')
plt.tight_layout(); plt.show()
print('Centros HSV [H, S, V]:')
print(np.round(centros, 1))

## 5. Análisis y aplicaciones

Responde para cada imagen:

1. ¿Qué estructuras aparecen en Canny y cuáles son ruido o textura?
2. ¿Otsu separa razonablemente el objeto y el fondo?
3. ¿La umbralización adaptativa mejora zonas con iluminación desigual?
4. ¿Es correcto elegir siempre el grupo más oscuro en k-means?
5. ¿Qué cambió después de la apertura y el cierre?
6. ¿Qué método elegirías para radiografía, frutas, `carta.jpg`, inspección industrial o una imagen satelital?

Aplicaciones posibles:

- radiografía: localizar estructuras óseas o regiones de interés;
- frutas: contar objetos, segmentar por color y extraer descriptores;
- `carta.jpg`: detectar regiones, bordes y posibles caracteres;
- satélite: separar agua, vegetación, suelo y zonas urbanas;
- industria: localizar grietas, piezas o defectos.

La segmentación es una etapa de apoyo. En aplicaciones médicas o industriales se necesita validar el resultado con datos etiquetados y criterios del dominio.